# 🚨 Crisis Negotiator — GRPO Training Notebook

**Multi-Agent De-escalation RL Environment** | OpenEnv Hackathon Submission

This notebook trains a Qwen2.5-7B model to act as an FBI-style crisis negotiator using:
- **GRPO** (Group Relative Policy Optimization) from HuggingFace TRL
- **Unsloth** 4-bit quantization for free Colab T4 GPU
- **Adaptive curriculum** that auto-escalates difficulty (easy → medium → hard)
- **Self-improvement**: failure-adaptive scenario mutation generates harder variants of failed episodes
- **Multi-component reward**: outcome, FBI technique usage, token efficiency (Mercor), safety oversight
- **Expert-in-the-loop** (Snorkel AI): rotating expert feedback with changing requirements

**Requirements**: Colab T4 GPU (free tier) or local GPU with ≥16GB VRAM

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl openenv-core peft accelerate datasets sentence-transformers
!pip install -q openai fastapi uvicorn pydantic requests numpy matplotlib

In [ ]:
# Cell 2: Clone repo (skip if already local)
import os
if not os.path.exists('server'):
    !git clone https://github.com/Dinesh052/Test.git crisis-negotiator
    %cd crisis-negotiator

In [ ]:
# Cell 3: Load model with Unsloth 4-bit LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=4096,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    use_gradient_checkpointing="unsloth",
)
print(f"Model loaded: {model.config._name_or_path}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Cell 4: Initialize environment + curriculum + self-improvement
import sys, json, re
sys.path.insert(0, '.')

from server.environment import CrisisNegotiatorEnvironment
from server.scenario_generator import AdaptiveCurriculum, FailureAdaptiveGenerator
from models import NegotiatorAction

env = CrisisNegotiatorEnvironment()
curriculum = AdaptiveCurriculum(window=10, threshold=0.7)
failure_gen = FailureAdaptiveGenerator(failure_threshold=0.4)

# Verify environment works
obs = env.reset(task_id='easy_domestic_desperate', seed=42)
obs = env.step(NegotiatorAction(
    action_type='emotional_label',
    content="It sounds like you're feeling overwhelmed.",
    reasoning='test', target='hostage_taker'
))
print(f"Environment OK: step reward={obs.reward:.3f}")
print(f"Curriculum: {curriculum.stats}")
print(f"Self-improvement: {failure_gen.stats}")

In [ ]:
# Cell 5: Define system prompt and helpers

SYSTEM_PROMPT = """You are an expert FBI-trained crisis negotiator. De-escalate the situation using:
- Emotional Labeling: "It sounds like you're feeling..."
- Mirroring: Repeat their last words
- Open Questions: "Tell me more about..."
- Demand Acknowledgment: Acknowledge without committing
- Stay calm. Never threaten. Never dismiss demands.

Respond with ONE JSON object:
{"action_type": "emotional_label|mirror|open_question|acknowledge_demand|speak|offer_concession|buy_time", "content": "your words", "reasoning": "your strategy", "target": "hostage_taker"}"""


def build_prompt(obs) -> str:
    parts = [f"Scenario: {obs.scenario_brief}"]
    parts.append(f"Step {obs.step}, Time left: {obs.time_remaining}")
    if obs.dialogue_history:
        for e in obs.dialogue_history[-4:]:
            parts.append(f"{e['speaker'].upper()}: {e['content']}")
    if obs.stated_demands:
        parts.append(f"Demands: {[d['text'] for d in obs.stated_demands]}")
    if obs.commander_patience != 'patient':
        parts.append(f"Commander: {obs.commander_patience}")
    return '\n'.join(parts)


def parse_action(text: str) -> dict:
    text = re.sub(r'```(?:json)?\s*', '', text.strip())
    text = re.sub(r'```\s*$', '', text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r'\{[^{}]*\}', text)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
    return {'action_type': 'speak', 'content': text[:200], 'reasoning': '', 'target': 'hostage_taker'}

print("Helpers defined.")

In [ ]:
# Cell 6: GRPO reward function with self-improvement

def run_episode(model_output: str, scenario_seed: int) -> float:
    """Run one episode and return terminal grader score."""
    scenario = curriculum.get_scenario(seed=scenario_seed)
    obs = env.reset(task_id='generate:' + scenario['difficulty'], seed=scenario_seed)

    action_dict = parse_action(model_output)
    try:
        action = NegotiatorAction(**action_dict)
    except Exception:
        action = NegotiatorAction(action_type='speak', content=model_output[:200],
                                 reasoning='', target='hostage_taker')

    obs = env.step(action)

    # Continue with heuristic policy for remaining steps
    for _ in range(20):
        if obs.done:
            break
        action = NegotiatorAction(
            action_type='emotional_label',
            content="I hear you. Tell me more about how you're feeling.",
            reasoning='maintain rapport', target='hostage_taker',
        )
        obs = env.step(action)

    # Use terminal grader score (NOT max of step rewards)
    final_reward = obs.reward if obs.done else max(0.01, min(0.99, 0.3))

    # Self-improvement: curriculum + failure-adaptive mutation
    curriculum.record(scenario.get('difficulty', 'medium'), final_reward)
    failure_gen.on_episode_end(scenario, final_reward, seed=scenario_seed)
    return final_reward


from typing import List

def reward_fn(completions: List[str], **kwargs) -> List[float]:
    """GRPO reward function — runs each completion through the environment."""
    rewards = []
    for i, completion in enumerate(completions):
        try:
            r = run_episode(completion, scenario_seed=i)
        except Exception:
            r = 0.01
        rewards.append(r)
    return rewards

print("Reward function defined.")

In [ ]:
# Cell 7: Display the reward curve
from IPython.display import Image
Image('reward_curve.png')

In [ ]:
# Cell 8: Summary stats
import json
data = json.load(open('reward_log.json'))
rewards = [d['reward'] for d in data]
first_10 = rewards[:10]
last_10 = rewards[-10:]

print(f'Episodes: {len(data)}')
print(f'First 10 avg reward: {sum(first_10)/10:.3f}')
print(f'Last 10 avg reward:  {sum(last_10)/10:.3f}')
print(f'Improvement: {sum(last_10)/10 - sum(first_10)/10:+.3f}')
print(f'Overall success rate: {sum(1 for r in rewards if r>0.5)/len(rewards):.0%}')

# Oversight stats
unsafe = sum(1 for d in data if d['outcome'] in ['harm_event','tactical_intervention','supervisor_termination'])
predicted = sum(1 for d in data if d.get('oversight_f1', 0) > 0)
print(f'\nOversight: predicted {predicted}/{unsafe} dangerous situations')

In [ ]:
# Cell 9: Download artifacts
from google.colab import files
files.download('reward_curve.png')
files.download('reward_log.json')